In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/cleaned_data.csv")
df.head()

,customerid,tenure,monthlycharges,totalcharges,contract,paymentmethod,paperlessbilling,seniorcitizen,churn
0,0,6,64,1540,1,1,0,1,0
1,1,21,113,1753,0,2,1,1,0
2,2,27,31,1455,2,1,0,1,0
3,3,53,29,7150,0,2,0,1,0
4,4,16,185,1023,1,2,0,1,0


In [3]:
df.describe()

,customerid,tenure,monthlycharges,totalcharges,contract,paymentmethod,paperlessbilling,seniorcitizen,churn
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000
mean,249.500000,36.532000,113.636000,4237.882000,0.948000,1.00800,0.486000,0.498000,0.106000
std,144.481833,20.667057,51.799903,2260.619837,0.791549,0.80326,0.500305,0.500497,0.308146
min,0.000000,1.000000,20.000000,159.000000,0.000000,0.00000,0.000000,0.000000,0.000000
25%,124.750000,19.000000,67.000000,2237.250000,0.000000,0.00000,0.000000,0.000000,0.000000
50%,249.500000,37.000000,115.000000,4182.500000,1.000000,1.00000,0.000000,0.000000,0.000000
75%,374.250000,54.000000,158.000000,6266.750000,2.000000,2.00000,1.000000,1.000000,0.000000
max,499.000000,71.000000,199.000000,7992.000000,2.000000,2.00000,1.000000,1.000000,1.000000


In [4]:
churn_rate = df['churn'].value_counts(normalize=True) * 100
print("Churn Rate (%):\n", churn_rate)

Churn Rate (%):
 churn
0    89.4
1    10.6
Name: proportion, dtype: float64


In [5]:
grouped = df.groupby('churn').mean()
grouped

,customerid,tenure,monthlycharges,totalcharges,contract,paymentmethod,paperlessbilling,seniorcitizen
churn,,,,,,,,
0,249.212528,40.152125,111.722595,4234.577181,0.997763,0.993289,0.483221,0.501119
1,251.924528,6.000000,129.773585,4265.754717,0.528302,1.132075,0.509434,0.471698


In [6]:
difference = grouped.loc[1] - grouped.loc[0]
difference.sort_values(ascending=False)

totalcharges        31.177536
monthlycharges      18.050990
customerid           2.712000
paymentmethod        0.138787
paperlessbilling     0.026212
seniorcitizen       -0.029420
contract            -0.469461
tenure             -34.152125
dtype: float64

In [7]:
correlation = df.corr()['churn'].sort_values(ascending=False)
correlation

churn               1.000000
monthlycharges      0.107381
paymentmethod       0.053241
paperlessbilling    0.016145
customerid          0.005784
totalcharges        0.004250
seniorcitizen      -0.018114
contract           -0.182759
tenure             -0.509208
Name: churn, dtype: float64

In [10]:
if 'usage' in df.columns:
    high_usage = df[df['usage'] > df['usage'].mean()]
    low_usage = df[df['usage'] <= df['usage'].mean()]

    print("High Usage Churn Rate:", high_usage['churn'].mean())
    print("Low Usage Churn Rate:", low_usage['churn'].mean())

In [9]:
for col in df.columns:
    if col != 'churn':
        print(f"\nAverage {col} for churned vs non-churned:")
        print(df.groupby('churn')[col].mean())


Average customerid for churned vs non-churned:
churn
0    249.212528
1    251.924528
Name: customerid, dtype: float64

Average tenure for churned vs non-churned:
churn
0    40.152125
1     6.000000
Name: tenure, dtype: float64

Average monthlycharges for churned vs non-churned:
churn
0    111.722595
1    129.773585
Name: monthlycharges, dtype: float64

Average totalcharges for churned vs non-churned:
churn
0    4234.577181
1    4265.754717
Name: totalcharges, dtype: float64

Average contract for churned vs non-churned:
churn
0    0.997763
1    0.528302
Name: contract, dtype: float64

Average paymentmethod for churned vs non-churned:
churn
0    0.993289
1    1.132075
Name: paymentmethod, dtype: float64

Average paperlessbilling for churned vs non-churned:
churn
0    0.483221
1    0.509434
Name: paperlessbilling, dtype: float64

Average seniorcitizen for churned vs non-churned:
churn
0    0.501119
1    0.471698
Name: seniorcitizen, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [12]:
X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.97


E:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
importance = pd.Series(model.coef_[0], index=X.columns)
importance.sort_values(ascending=False)

paymentmethod       0.617413
paperlessbilling    0.252215
seniorcitizen       0.024330
monthlycharges      0.018293
totalcharges        0.000218
customerid         -0.001186
tenure             -0.330365
contract           -1.542870
dtype: float64